# Creating a Simple Agent with Tracing

In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [2]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Codeing Assistant Agent

In [4]:
Bug_Hunter = Agent(
    name="BugHunter",
    instructions="""
    You are a Python Bug Hunter.
    Find errors in code, explain why they occur,
    and suggest corrected code.
    Keep answers concise.
    """
)

Let's execute the Agent:

In [7]:
result = await Runner.run(
    Bug_Hunter,
    'x = 10\ny = "5"\nprint(x + y)\nWhich error and how to correct it?'
)

print(result)

RunResult:
- Last agent: Agent(name="BugHunter", ...)
- Final output (str):
    Error: TypeError: unsupported operand type(s) for +: 'int' and 'str'
    
    Fix by converting types:
    - If you want numeric addition: print(x + int(y))
    - If you want string concatenation: print(str(x) + y)
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [9]:
response_stream = Runner.run_streamed(
    Bug_Hunter,
    '''Find bugs in this code and explain how to fix them:
    x = 10
    y = "5"
    print(x + y)
    '''
)


async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Issue: TypeError from adding int and str.

Cause: x is int (10) and y is str ("5"); Python cannot add different types.

Fix options:
- Convert y to int: print(x + int(y))  # 15
- Convert x to str for concatenation: print(str(x) + y)  # "105"
- Use safe parsing (handle non-numeric): 
  try:
      print(x + int(y))
  except ValueError:
      print("y is not a number")

Minimal corrected code example (int addition):
x = 10
y = "5"
print(x + int(y))  # 15

_Good Job!_